# Imports & Raw Data

In [ ]:
from geosupport import Geosupport # Used for making calls to NYC Geoservice API
from geosupport import GeosupportError
import pandas as pd #used for reading and writing csvs as dataframe objects
import usaddress as addy # used for address parsing
import numpy as np
import re

#Path for Geosupport Desktop edition. Required for using python library.
g = Geosupport(geosupport_path=r"C:\Program Files\Geosupport Desktop Edition") 
#Raw data
df = pd.read_csv(r"C:\Users\address input.csv", encoding='UTF-8') 
#creating a set (unique list) of NYC Zipcodes
nyc_zip = set(pd.read_csv(r"C:\Users\nyc-zip-codes.csv", encoding='UTF-8').ZipCode) 

result ={} #Empty Dictionary for storing responses to calls. City Council district can be retrieved using the command result['City Council District']
adr_list = []

#Defining Borough dictionary to map onto the County Column
Borough = {
    "kings county":"brooklyn","kings":"brooklyn","brooklyn":"brooklyn",
    "bronx county":"bronx","bronx":"bronx","the bronx":"bronx",
    "new york county":"manhattan","new york":"manhattan","manhattan":"manhattan",
    "queens county":"queens","queens":"queens",
    "richmond county":"staten island","richmond":"staten island","staten island":"staten island"}
#Defining code dictionary to map onto the Program Assigned Column
codes = {"Administration/Management":'202',"Adult Education":'201',"Anti-Violence Program":'199',"Bonding and Play Circle":'3042',
         "Emergency Relief Fund":'44738',"Health Benefits Navigation Program":'28977',"Health Insurance Program":'1069',
         "Housing Navigation Program":'136503',"Kensignton Arts Program":'3688',"Legal Services":'3913',"Limited Access":'10498',
         "Mental Health Counseling Program":'11845',"Outreach & External Engagement":'75091',"Preventive Services":'1056',
         "Resource Development":'65163',"Supplemental Nutrition Assistance Program SNAP":'233',"Young Adult Youth":'200',
         "PHS-WholeYouNYC":"236508","Emergency Response Program":"253347"}

# Some basic data cleaning
df['Line1'] = df['Line1'].astype(str) #Line1 as string
df['County'] = df['County'].astype(str) #County as string
df['City Council District'] = df['City Council District'].astype(object) #CCD as new column
df['Assigned Programs'] = df['Assigned Programs'].astype(str)
df['Line1'] = df['Line1'].str.lower() #lower case for simplicity
df['County'] = df['County'].str.lower() #lower case for simplicity
df['Assigned Programs'] = df['Assigned Programs'].str.replace(',', '|').astype('str') #necessary for future import
df['Assigned Programs'] = df['Assigned Programs'].str.replace('(', '').astype('str') #neccessary to avoid issues with ( in the replace method
df['Assigned Programs'] = df['Assigned Programs'].str.replace(')', '').astype('str') #neccessary to avoid issues with ( in the replace method

#Mapping borough dictionary for a new borough column
df['Borough'] = df['County'].map(Borough)
#Mapping codes dictionary for new Program Assigned column
df['New Assigned Programs'] = df['Assigned Programs'].replace(codes, regex=True)

print("total rows in raw data is", len(df))

# Separating Valid & Invalid Addresses

In [ ]:
df_valid = pd.DataFrame() #Empty dataframes
df_invalid = pd.DataFrame() # Empty dataframes

def valid_address(df):
    global df_valid
    global df_invalid
    # Step1. For each row in df if, zipcode is in the set add to good_zip else add to bad_zip
    for row in list(df.index):
        if (df.loc[row, 'Zip'] in nyc_zip) == True:
            new_row = df.iloc[row].to_frame().T
            df_valid = pd.concat([df_valid, new_row], ignore_index=True)
        elif (df.loc[row, 'Zip'] in nyc_zip) == False:
                if pd.isna(df.at[row,'Zip']):
                    df.at[row,'City Council District'] = "Unknown"
                    new_row = df.iloc[row].to_frame().T
                    df_invalid = pd.concat([df_invalid, new_row], ignore_index=True)
                else:
                    df.at[row,'City Council District'] = "Not Applicable (Non-NYC Resident)"
                    new_row = df.iloc[row].to_frame().T
                    df_invalid = pd.concat([df_invalid, new_row], ignore_index=True)
    
    #Step 2. Applying mask for unusable addresses but valid zipcodes
    mask = ((df_valid['Borough'].isnull()) | (df_valid['Line1'].isnull()) | (df_valid['Line1'].str.contains(r'^\D'))) # mask for blank borough or blank address or
    opposite_mask = ~mask # opposite of the previous mask
    df1 = df_valid[mask]
    df1 = df1.reset_index(drop=True)
    #Step 4. For each row flagged by the filter use concat to add it to Bad zip dataframe with City Council District = Unknown 
    for row in list(df1.index):
        df1.loc[row,'City Council District'] = "Unknown"
        new_row = df1.iloc[row].to_frame().T
        df_invalid = pd.concat([df_invalid, new_row], ignore_index=True)
    
    # Step 5. Data cleaning
    df_invalid['County'] = df_invalid['County'].str.title() #capitalizing
    df_invalid['Line1'] = df_invalid['Line1'].str.title() #capitalizing
    df_invalid['County'] = df_invalid['County'].str.replace('Nan', '', regex=True).astype('str') #removing Nan values after capitalizing
    df_invalid['Line1'] = df_invalid['Line1'].str.replace('Nan', '', regex=True).astype('str') #removing Nan values after capitalizing

    df_invalid.to_csv(r"C:\Users\invalid.csv", index = True, header=True) # Saving invalid addresses
    df_valid = df_valid[opposite_mask]

#Calling the function on the raw data
valid_address(df)

#Quality check on number of rows
print("length of df_invalid is", len(df_invalid))
print("length of df_valid is", len(df_valid))
print("total rows are", len(df_invalid) + len(df_valid))

In [ ]:
df_valid.to_csv(r"C:\Users\df_valid.csv", index = True, header=True) #Saving valid addresses for possible review

# Cleaning & Parsing Valid Addresses

In [ ]:
df_clean = df_valid
df_clean['Line1'] = df_clean['Line1'].astype(str)
#removing hyphens in Housenumbers.
df_clean['Line1'] = df_clean['Line1'].str.replace(r'-(?=[0-9]{2})', '', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'-(?=[0-9]{1})', '0', regex=True).astype('str')
#Removing number modifiers
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)street', ' street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)\s{0}(st)', '', regex=True).astype('str')
#df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)street', ' street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)\s{0,1}(rd)', '', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)\s{0}(th)', '', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)\s{0,1}(nd)', '', regex=True).astype('str')
#removing street type abbreviations
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ave)(?=\s)', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ave)$', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ave.)(?=\s)', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ave.)$', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(av)(?=\s)', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(av)$', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(av.)(?=\s)', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(av.)$', 'avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(st)(?=\s)', 'street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(st.)(?=\s)', 'street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(st)$', 'street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(st.)$', 'street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(rd)(?=\s)', 'road', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(rd)$', 'road', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(blvd)(?=\s)', 'boulevard', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(blvd)$', 'boulevard', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ln)(?=\s)', 'lane', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(ln)$', 'lane', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(pl)(?=\s)', 'place', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(pl)$', 'place', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(dr)(?=\s)', 'drive', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\s)(dr)$', 'drive', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(dr)$', 'drive', regex=True).astype('str')

df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)place', ' place', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)drive', ' drive', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)street', ' street', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)avenue', ' avenue', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)road', ' road', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)boulevard', ' boulevard', regex=True).astype('str')
df_clean['Line1'] = df_clean['Line1'].str.replace(r'(?<=\d)lane', ' lane', regex=True).astype('str')
#remove extra spaces with a single space
df_clean['Line1'] = df_clean['Line1'].str.replace(r'\s{2,}', ' ', regex=True).astype('str')

for i in list(df_clean.index):
#First we will parse the address into it's relevant parts
#First step will be to separate the house number from the rest of the address using the usaddress library from above
    try:
        adr = addy.parse(df_clean.loc[i, 'Line1']) # Line1 is the column with the street address
    except:
        pass
    df_clean.loc[i, 'House Number'] = (adr[0][0])
    if len(adr) == 5:
        df_clean.loc[i, 'Street Name'] = adr[1][0] + ' ' + adr[2][0] + ' ' + adr[3][0] + ' ' + adr[4][0]
    if len(adr) == 4:
        df_clean.loc[i, 'Street Name'] = adr[1][0] + ' ' + adr[2][0] + ' ' + adr[3][0]
    elif len(adr) == 3:
        df_clean.loc[i, 'Street Name'] = adr[1][0] + ' ' + adr[2][0]
    elif len(adr) == 2:
        df_clean.loc[i, 'Street Name'] = adr[1][0]
    else:
        pass 

In [ ]:
df_clean.to_csv(r"C:\Users\parsed.csv", index = True, header=True) #Saving parsed addresses for possible review

## GeoSupport Calls

In [ ]:
# from geosupport import Geosupport # Used for making calls to NYC Geoservice API
from geosupport import Geosupport 
from geosupport import GeosupportError
import pandas as pd #used for reading and writing csvs as dataframe objects
import numpy as np
import re

#g = Geosupport(geosupport_path=r"C:\Program Files\Geosupport Desktop Edition")
df_clean = pd.read_csv(r"C:\Users\parsed.csv", encoding='UTF-8') #input address data. Dataframe object
nta = pd.read_csv(r"C:\Users\omar\OneDrive - Arab American Family Support\Desktop\NTA codes.csv", encoding='UTF-8') #List of Neighborhood codes and their names
nta = nta.reset_index(drop=True)
NTA = nta.set_index('NTACode')['NTAName'].to_dict()

# Function Start - Get Similar Street Names
def get_similar_streets(error):
    """Extract similar street names from a GeosupportError."""
    result = error.result
    # Get count of similar names (may be string or int)
    count = result.get("Number of Street Codes and Street Names in List", 0)
    if isinstance(count, str):
        count = int(count.strip() or "0")
    if count == 0:
        return []
    # Get list of street names (already parsed into a list by the library)
    similar_names = result.get("List of Street Names", [])
    # Filter out empty strings and return
    return [name for name in similar_names if name and name.strip()]  
# Function End

# Function - GeoSupport Calls
## Note: Borough code can be mn, bk, bx, qn, si. borough names are also accepted
def geo_call(df_clean):
    for i in list(df_clean.index):
        try:
            result = g.address(house_number = int(df_clean.loc[i, 'House Number']), street_name = df_clean.loc[i, 'Street Name'] , borough_code=df_clean.loc[i,'Borough'])
        except ValueError as v:
            error = str(v)
            df_clean.loc[i,'Errors'] = v
        except GeosupportError as geo:
            error = str(geo)
            df_clean.loc[i,'Errors'] = geo
###############            
            print(f"Error: {geo}\n")
            # Extract and display similar names
            similar = get_similar_streets(geo)
            #If error is has n similar names
            if similar:
                print(f"Found {len(similar)} similar street(s):")
                print(f"For context original address is", df_clean.loc[i, 'Geolocation'])
                for x, name in enumerate(similar, 1):
                    print(f"  {x}. {name}")
                    #Pause for user input to select the proper street name
                choice = input("Please select the number from the list of 10 similar streets to retry. To pass this row enter any letter instead")
                if choice.isdigit():
                    choice = int(choice) - 1
                    print(f"\nRetrying with user input: '{similar[choice]}'")
                    try:
                        result_sim = g.address(house_number = df_clean.loc[i, 'House Number'], street_name = similar[choice] , borough_code=df_clean.loc[i,'Borough'])
                        
                        df_clean.loc[i, 'Street Name'] = similar[choice]
                        df_clean.loc[i, 'New CD'] = result_sim['City Council District']
                        df_clean.loc[i, 'Assembly District'] = result_sim['Assembly District']
                        df_clean.loc[i, 'State Senatorial District'] = result_sim['State Senatorial District']
                        df_clean.loc[i, 'Congressional District'] = result_sim['Congressional District']
                        df_clean.loc[i, "New Zip"] = result_sim['ZIP Code']
                        df_clean.loc[i, "NTA Code"] = result_sim['Neighborhood Tabulation Area (NTA)']
                        df_clean.loc[i,'Errors'] = "fixed"
                        print(f"Success! values entered")
                    except:
                        print("That didn't work, please check manually")
                elif not(choice.isdigit()):
                    print("passing this row")
                else:
                    pass
            pass
###########        
        else:
            df_clean.loc[i, 'New CD'] = result['City Council District']
            df_clean.loc[i, 'Assembly District'] = result['Assembly District']
            df_clean.loc[i, 'State Senatorial District'] = result['State Senatorial District']
            df_clean.loc[i, 'Congressional District'] = result['Congressional District']
            df_clean.loc[i, "New Zip"] = result['ZIP Code']
            df_clean.loc[i, "NTA Code"] = result['Neighborhood Tabulation Area (NTA)']
            
    df_clean['Street Name'] = df_clean['Street Name'].str.title() #lower case for simplicity
    df_clean['County'] = df_clean['County'].str.title() #lower case for simplicity
    df_clean['House Number'] = df_clean['House Number'].astype(str) #Line1 as string
    df_clean["New Line1"] = df_clean['House Number'].str.cat(df_clean['Street Name'], sep=' ')
    df_clean['Neighborhood'] = df_clean['NTA Code'].replace(NTA)
    df_clean.to_csv(r"C:\Users\output.csv", encoding='UTF-8')
    

geo_call(df_clean)

### For testing specific rows

In [ ]:
from geosupport import Geosupport 
from geosupport import GeosupportError

nta2 = nta.set_index(nta.columns[0])
g = Geosupport(geosupport_path=r"C:\Program Files\Geosupport Desktop Edition")
#df_clean = pd.read_csv(r"C:\Users\omar\OneDrive - Arab American Family Support\GeoSupport steps\December 2025\Address Parse_12.2.25.csv", encoding='UTF-8')
# Function Start - Get Similar Street Names
def get_similar_streets(error):
    """Extract similar street names from a GeosupportError."""
    result = error.result
    # Get count of similar names (may be string or int)
    count = result.get("Number of Street Codes and Street Names in List", 0)
    if isinstance(count, str):
        count = int(count.strip() or "0")
    if count == 0:
        return []
    # Get list of street names (already parsed into a list by the library)
    similar_names = result.get("List of Street Names", [])
    # Filter out empty strings and return
    return [name for name in similar_names if name and name.strip()]   
# Function End

# Function - GeoSupport Calls
## Note: Borough code can be mn, bk, bx, qn, si. borough names are also accepted
def geo_call (housenumber, street_name, borough_code):    
    try:
        result = g.address(house_number = housenumber, street_name = street_name , borough_code = borough_code)
    except GeosupportError as geo:
        print(f"Error: {geo}\n")
        # Extract and display similar names
        similar = get_similar_streets(geo)
        if similar:
            print(f"Found {len(similar)} similar street(s):")
            for i, name in enumerate(similar, 1):
                print(f"  {i}. {name}")
            #How to Proceed
            question_1 = input("If you want to choose an option from the list enter the number otherwise enter enter any letter to continue")
            if question_1.isdigit():
                question_1 = question_1 - 1
                # You could now retry with similar[0] or let user choose
                print(f"\nRetrying with list option: '{similar[choice]}'")
                result = g.address(house_number = housenumber, street_name = similar[choice] , borough_code = borough_code)
                print(f"Success! Results are")
                print('City Council District is',result['City Council District'])
                print('State Assembley District is',result['Assembly District'])
                print('State Assembly District is',result['State Senatorial District'])
                print('Congressional District is',result['Congressional District'])
                print('NTA Code is',result['Neighborhood Tabulation Area (NTA)'])
                print('Zip Code is',result['ZIP Code'])
            else:
                print("try again")
    else:
        print('City Council District:',result['City Council District'])
        print('State Assembley District:',result['Assembly District'])
        print('State Senate District:',result['State Senatorial District'])
        print('Congressional District:',result['Congressional District'])
        print('NTA Code:',result['Neighborhood Tabulation Area (NTA)'])
        print('Neighborhood:', (nta2.loc[result['Neighborhood Tabulation Area (NTA)'],"NTAName"]))
        print('Zip Code is',result['ZIP Code'])
# Function End        

In [ ]:
geo_call("270",'73 street','bk')